# Möwenschwärme – erkunden

VSIT-50 · Vorlesung „Mutationen in Distributed Systems“

Führen Sie die Zellen der Reihe nach aus. Zwischen den Abschnitten stehen Aufgaben mit leeren Zellen.
Diese Übung wird nicht bewertet – sie dient Ihrer Übung.

## 1 — Das Quiz

`flock.py` enthält die Klasse aus der Vorlesung. Schwarm A hat 4 Möwen, B hat 2, C hat 0.

```python
result = (flock_a.conjoin(flock_c)
                 .breed(flock_b)
                 .conjoin(flock_a.breed(flock_b))
                 .seagulls)
```

### Ihre Aufgabe 1

Rechnen Sie das Ergebnis **im Kopf** aus, bevor Sie die nächste Zelle ausführen. Notieren Sie Ihre Zahl.

In [ ]:
# Meine Vorhersage:
vorhersage = None

In [ ]:
import importlib

import flock

importlib.reload(flock)
print("Vorhersage:", vorhersage, "  Python:", flock.quiz())

## 2 — Was hinter dem Namen passiert

`id()` liefert die Identität eines Objekts. Die Zelle verfolgt den Ausdruck Schritt für Schritt.

In [ ]:
from flock import Flock

importlib.reload(flock)
Flock = flock.Flock

flock_a, flock_b, flock_c = Flock(4), Flock(2), Flock(0)
print("flock_a      ", id(flock_a), flock_a.seagulls)

s1 = flock_a.conjoin(flock_c)
print("nach conjoin ", id(s1), s1.seagulls, "| flock_a:", flock_a.seagulls)

s2 = s1.breed(flock_b)
print("nach breed   ", id(s2), s2.seagulls, "| flock_a:", flock_a.seagulls)

arg = flock_a.breed(flock_b)
print("Argument     ", id(arg), arg.seagulls, "| flock_a:", flock_a.seagulls)

s3 = s2.conjoin(arg)
print("Ergebnis     ", id(s3), s3.seagulls)

### Ihre Aufgabe 2

a) Wie viele verschiedene `Flock`-Objekte gibt es während des ganzen Ausdrucks?

b) Erklären Sie in zwei Sätzen, warum 32 herauskommt und nicht 16. Verwenden Sie die Begriffe *Referenz* und *Zustand*.

In [ ]:
# Ihre Antwort (als Kommentar):

## 3 — Dasselbe Muster in Python selbst

Auch eingebaute Typen kennen beide Varianten: Methoden, die das Objekt ändern, und Funktionen, die einen neuen Wert liefern.

In [ ]:
zahlen = [3, 1, 2]
alias = zahlen

print(sorted(zahlen), zahlen)      # neuer Wert
print(alias.sort(), zahlen)        # ändert das Objekt, liefert None

### Ihre Aufgabe 3

Ordnen Sie zu – ändert der Aufruf das Objekt, oder liefert er einen neuen Wert?
`list.append`, `list + list`, `list += list`, `str.upper`, `dict.update`, `dict | dict`, `tuple + tuple`.
Prüfen Sie jede Vermutung mit `id()` vor und nach dem Aufruf.

Welcher Fall hat Sie überrascht?

In [ ]:
# Ihre Untersuchung:

## 4 — Den Fehler beheben

Ändern Sie `flock.py` so, dass `conjoin` und `breed` **keinen** Schwarm verändern, sondern einen neuen liefern.
Der Ausdruck in `quiz()` bleibt Zeichen für Zeichen gleich.

`tests/test_flock.py` ist die Spezifikation. Anfangs ist er rot.

In [ ]:
!python -m pytest -q tests/test_flock.py

### Ihre Aufgabe 4

a) Bringen Sie `tests/test_flock.py` auf grün und führen Sie die Zelle oben erneut aus.

b) Schreiben Sie das Quiz ohne Klasse, nur mit zwei Funktionen `conjoin(x, y)` und `breed(x, y)` auf Zahlen.
Was fällt Ihnen an den beiden Funktionen auf?

In [ ]:
# Ihre Lösung zu 4b:

## 5 — Mutation verbieten statt vermeiden

Disziplin allein hält nicht lange. `@dataclass(frozen=True)` lässt Python jede Zuweisung an ein Feld ablehnen.

In [ ]:
from dataclasses import FrozenInstanceError, dataclass


@dataclass(frozen=True)
class Size:
    n: int


s = Size(4)
try:
    s.n = 8
except FrozenInstanceError as e:
    print("abgelehnt:", e)

### Ihre Aufgabe 5

Schreiben Sie `Flock` als `frozen` Dataclass mit `conjoin` und `breed`.
Bauen Sie dann absichtlich die alte, mutierende Variante von `conjoin` ein. Was passiert – und wann: beim Definieren der Klasse oder beim Aufruf?

In [ ]:
# Ihre Flock-Dataclass:

## 6 — Ausbau: Schwärme mit Id

Jetzt soll man einen Schwarm über die Zeit verfolgen können. Bevor Sie Code schreiben, entscheiden Sie fachlich:

- Wenn sich Schwarm C an Schwarm A anschließt: Bleibt die Id von A? Was wird aus C – löschen, oder behalten mit einem Vermerk?
- Wenn A und B brüten: Ist das Ergebnis noch A, oder ein neuer Schwarm? Wovon hängt das ab?
- Ändert eine Id irgendetwas am Ergebnis 32?

`flock_ids.py` trennt **Identität** (`flock_id` – wer) von **Stand** (`version` – wann).
Jede Änderung erzeugt eine neue, unveränderliche `FlockVersion`.

### Ihre Aufgabe 6

Ergänzen Sie `conjoin`, `breed` und `Repository.save` in `flock_ids.py`, bis `tests/test_flock_ids.py` grün ist.
Lesen Sie zuerst die Tests – sie legen die fachlichen Entscheidungen fest.

In [ ]:
!python -m pytest -q tests/test_flock_ids.py

## 7 — Zwei Writer, ein Schwarm

In einem verteilten System lesen zwei Dienste denselben Stand A@0 und schreiben beide eine Version 1.
`Repository.save` lässt nur den ersten durch.

In [ ]:
import flock_ids

importlib.reload(flock_ids)
from flock_ids import FlockVersion, Repository, VersionConflict, conjoin

repo = Repository()
a0, c0 = FlockVersion("A", 0, 4), FlockVersion("C", 0, 3)
try:
    repo.save(a0)
    a1, _ = conjoin(a0, c0)
    repo.save(a1)                                  # Dienst 1
    repo.save(FlockVersion("A", 1, 8))             # Dienst 2, ebenfalls auf A@0
except VersionConflict as e:
    print("Konflikt:", e)
except NotImplementedError:
    print("Erst Aufgabe 6 lösen.")

### Ihre Aufgabe 7

a) Was stünde ohne Versionsprüfung („last writer wins“) am Ende im Repository? Welche Änderung ginge verloren?

b) Wie sollte Dienst 2 auf den `VersionConflict` reagieren? Beschreiben Sie die Schritte.

c) Wie würden Sie dieselbe Prüfung in SQL formulieren, mit einer einzigen `UPDATE`-Anweisung?

In [ ]:
# Ihre Antworten (als Kommentar):